# 03 - Comparativa de Detectores de Anomalías: Isolation Forest vs. LOF vs. One-Class SVM vs. Elliptic Envelope

## Curso de Machine Learning: Modelos No Supervisados

Scikit-Learn ofrece cuatro familias principales de algoritmos para detección no supervisada de valores atípicos y novedades. Cada una descansa sobre un supuesto matemático distinto:
1. **`EllipticEnvelope` (Gaussiano / Mahalanobis Robusto)**: Asume que todos los datos legítimos provienen de una única distribución normal multivariada elíptica.
2. **`OneClassSVM` (Frontera de Soporte con Kernel)**: Aprende un hiperplano o hipersuperficie envolvente alrededor de la región donde se concentra la masa de datos.
3. **`LocalOutlierFactor` (LOF - Densidad Local Relativa)**: Compara la densidad local de un punto con la densidad local de sus $k$ vecinos más cercanos.
4. **`IsolationForest` (Particionamiento Aleatorio en Árboles)**: Aísla observaciones mediante cortes ortogonales aleatorios.

En este cuaderno compararemos sus fronteras de decisión, tiempos de cómputo y casos de uso ideales en detección de fraude.

---

### Contenidos
1. Taxonomía de los 4 Detectores de Anomalías en Scikit-Learn
2. `EllipticEnvelope`: Estimación de Covarianza de Determinante Mínimo (MCD)
3. `LocalOutlierFactor` (LOF): Anomalías Locales vs. Globales (`novelty=True` vs `False`)
4. `OneClassSVM`: Fronteras No Lineales con Kernel RBF
5. Comparación Visual de Fronteras de Decisión ante Distintas Topologías de Datos
6. Benchmark Cuantitativo de Velocidad y Exactitud en Detección de Fraude
7. Conclusiones y Tabla de Selección de Arquitectura


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, average_precision_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Visualización Comparativa de Fronteras de Decisión en 2D

Evaluemos cómo traza su frontera de normalidad cada uno de los cuatro algoritmos sobre dos escenarios complejos:
- **Escenario 1**: Distribución bimodal (dos segmentos de clientes legítimos separados).
- **Escenario 2**: Variedad curva no lineal (*Moons*) contaminada con transacciones fraudulentas uniformes.


In [ ]:
rng = np.random.default_rng(42)
n_muestras = 350
frac_fraude = 0.10
n_fraude = int(n_muestras * frac_fraude)
n_norm = n_muestras - n_fraude

# Escenario 1: Bimodal
X1_norm, _ = make_blobs(n_samples=n_norm, centers=[[-2, -2], [2, 2]], cluster_std=0.7, random_state=42)
X1_out = rng.uniform(low=-6, high=6, size=(n_fraude, 2))
X1 = np.vstack([X1_norm, X1_out])

# Escenario 2: No lineal (Moons)
X2_norm, _ = make_moons(n_samples=n_norm, noise=0.08, random_state=42)
X2_norm = StandardScaler().fit_transform(X2_norm) * 1.8
X2_out = rng.uniform(low=-6, high=6, size=(n_fraude, 2))
X2 = np.vstack([X2_norm, X2_out])

modelos = [
    ('EllipticEnvelope (Gaussiano)', EllipticEnvelope(contamination=frac_fraude, random_state=42)),
    ('One-Class SVM (RBF)', OneClassSVM(nu=frac_fraude, kernel='rbf', gamma=0.25)),
    ('Local Outlier Factor (LOF)', LocalOutlierFactor(n_neighbors=25, contamination=frac_fraude, novelty=True)),
    ('Isolation Forest', IsolationForest(n_estimators=150, contamination=frac_fraude, random_state=42))
]

fig, axes = plt.subplots(2, 4, figsize=(19, 9))
xx, yy = np.meshgrid(np.linspace(-6.5, 6.5, 200), np.linspace(-6.5, 6.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

for row, (X_data, nombre_esc) in enumerate([(X1, 'Bimodal'), (X2, 'No Lineal')]):
    for col, (nombre_mod, mod) in enumerate(modelos):
        ax = axes[row, col]
        mod.fit(X_data)
        preds = mod.predict(X_data)
        Z = mod.decision_function(grid).reshape(xx.shape)
        
        ax.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap='Blues_r', alpha=0.4)
        ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors='crimson')
        ax.scatter(X_data[preds==1, 0], X_data[preds==1, 1], c='#1f77b4', s=20, edgecolors='k', alpha=0.8)
        ax.scatter(X_data[preds==-1, 0], X_data[preds==-1, 1], c='crimson', marker='x', s=45, lw=1.8)
        if row == 0:
            ax.set_title(nombre_mod, fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f'Escenario {nombre_esc}', fontsize=11, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Comparación de Fronteras de Normalidad (Línea Roja = Umbral de Alerta)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 2. Benchmark en un Problema de Fraude Multidimensional ($N = 10,000, P = 15$)

Comparemos el tiempo de ejecución y la precisión promedio (**PR-AUC**, la métrica más importante en fraude desbalanceado) sobre un conjunto de $10,000$ transacciones y $15$ variables.


In [ ]:
# Simulación de 10,000 transacciones en 15 dimensiones con 1.5% de fraude
X_bench_norm, _ = make_blobs(n_samples=9850, n_features=15, centers=3, cluster_std=1.5, random_state=42)
X_bench_fraud = rng.uniform(low=-10, high=10, size=(150, 15))
X_bench = StandardScaler().fit_transform(np.vstack([X_bench_norm, X_bench_fraud]))
y_bench_fraude = np.array([0]*9850 + [1]*150)

resultados_bench = []
for nombre, mod in [
    ('Isolation Forest', IsolationForest(n_estimators=100, random_state=42, n_jobs=-1)),
    ('Local Outlier Factor (LOF)', LocalOutlierFactor(n_neighbors=20, novelty=True, n_jobs=-1)),
    ('One-Class SVM (RBF)', OneClassSVM(nu=0.02, kernel='rbf')),
    ('Elliptic Envelope', EllipticEnvelope(contamination=0.015, random_state=42))
]:
    t0 = time.perf_counter()
    mod.fit(X_bench)
    scores = -mod.decision_function(X_bench)
    t_total = time.perf_counter() - t0
    
    resultados_bench.append({
        'Algoritmo': nombre,
        'Tiempo_Segundos': round(t_total, 3),
        'ROC_AUC': round(roc_auc_score(y_bench_fraude, scores), 4),
        'PR_AUC (Average Precision)': round(average_precision_score(y_bench_fraude, scores), 4)
    })

df_bench = pd.DataFrame(resultados_bench).sort_values('Tiempo_Segundos')
print("Benchmark en N = 10,000 transacciones y P = 15 dimensiones:")
display(df_bench)


## 3. Conclusiones y Guía de Selección

| Algoritmo | Complejidad | ¿Requiere Escalar? | Fortaleza Principal | Limitación |
| :--- | :---: | :---: | :--- | :--- |
| **`IsolationForest`** | $O(T \cdot \psi \log \psi)$ | **No** | Ultra rápido en Big Data y alta dimensión; ideal para fraude tabular. | Cortes paralelos a los ejes pueden crear artefactos fantasma en 2D. |
| **`LocalOutlierFactor`** | $O(n^2)$ | **Sí** | Detecta anomalías locales sutiles junto a clusters muy densos. | Lento en inferencia y sensible a la maldición de la dimensionalidad. |
| **`OneClassSVM`** | $O(n^2)$ a $O(n^3)$ | **Sí** | Excelente ajustando variedades curvas limpias. | Muy sensible a outliers presentes durante el entrenamiento y lento en $N$ grande. |
| **`EllipticEnvelope`** | $O(n \cdot p^3)$ | Recomendado | Óptimo si los datos son estrictamente Gaussianos unimodales. | Fracasa si existen múltiples segmentos de clientes (multimodal). |
